In [1]:
from sklearn.svm import SVR
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import root_mean_squared_error,r2_score

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")

In [3]:
data = pd.read_csv(r"E:\ML Datasets\ai4i2020.csv")

In [4]:
data

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,M24855,M,298.8,308.4,1604,29.5,14,0,0,0,0,0,0
9996,9997,H39410,H,298.9,308.4,1632,31.8,17,0,0,0,0,0,0
9997,9998,M24857,M,299.0,308.6,1645,33.4,22,0,0,0,0,0,0
9998,9999,H39412,H,299.0,308.7,1408,48.5,25,0,0,0,0,0,0


In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   UDI                      10000 non-null  int64  
 1   Product ID               10000 non-null  object 
 2   Type                     10000 non-null  object 
 3   Air temperature [K]      10000 non-null  float64
 4   Process temperature [K]  10000 non-null  float64
 5   Rotational speed [rpm]   10000 non-null  int64  
 6   Torque [Nm]              10000 non-null  float64
 7   Tool wear [min]          10000 non-null  int64  
 8   Machine failure          10000 non-null  int64  
 9   TWF                      10000 non-null  int64  
 10  HDF                      10000 non-null  int64  
 11  PWF                      10000 non-null  int64  
 12  OSF                      10000 non-null  int64  
 13  RNF                      10000 non-null  int64  
dtypes: float64(3), int64(9)

In [6]:
data.isnull().sum()

UDI                        0
Product ID                 0
Type                       0
Air temperature [K]        0
Process temperature [K]    0
Rotational speed [rpm]     0
Torque [Nm]                0
Tool wear [min]            0
Machine failure            0
TWF                        0
HDF                        0
PWF                        0
OSF                        0
RNF                        0
dtype: int64

In [7]:
# deleting uid column
data = data.drop(columns="UDI")

In [8]:
data

,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,M24855,M,298.8,308.4,1604,29.5,14,0,0,0,0,0,0
9996,H39410,H,298.9,308.4,1632,31.8,17,0,0,0,0,0,0
9997,M24857,M,299.0,308.6,1645,33.4,22,0,0,0,0,0,0
9998,H39412,H,299.0,308.7,1408,48.5,25,0,0,0,0,0,0


In [9]:
# setting faeture and target
x = data.drop(columns="Machine failure")
y = data["Machine failure"]

In [10]:
# train test split
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.3,random_state=42)
print("Train Size :",x_train.shape)
print("Test Size :",x_test.shape)

Train Size : (7000, 12)
Test Size : (3000, 12)


In [ ]:
# identifying numerical and categorical column
num_cols = []
cat_cols = data.select_dtypes(include="object").columns.tolist()

for i in data.columns:
    if data[i].dtypes != "object":
        if i == "Machine failure":
            continue
        else:
            num_cols.append(i)

print("Numerical Features :", num_cols)
print("Categorical Features :", cat_cols)

In [ ]:
# creating preprocessor pipeline
num_transformer = Pipeline(steps=[
    ("scaler",StandardScaler())
])
cat_transformer = Pipeline(steps=[
    ("onehot",OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer(transformers=[
    ("num",num_transformer,num_cols),
    ("cat",cat_transformer,cat_cols)
])

In [ ]:
preprocessor

In [ ]:
# creating baseline model
base_model = Pipeline(steps=[
    ("preprocessor",preprocessor),
    ("svr",SVR())
])
base_model.fit(x_train,y_train)

In [ ]:
# predict and evaluate model
y_pred = base_model.predict(x_test)
rmse = root_mean_squared_error(y_test,y_pred)
r2 = r2_score(y_test,y_pred)

print("Model Evaluation")
print("Root Mean Squared Error :", round(rmse,4))
print("R2 Score :", round(r2,4))

In [ ]:
# model hupertuning
parm_grid = {
    "svr__C" : [1,3,5,7,10,30,50,75],
    "svr__kernel" : ["linear","rbf","poly","sigmoid"],
    "svr__gamma" : [None,"auto","scale"],
    "svr__max_iter" : [-1,1000,2000,3000]
}

grid = GridSearchCV(base_model,param_grid=parm_grid,scoring="r2",cv=3,n_jobs=1,verbose=2)
grid.fit(x_train,y_train)

print("Best Parameter :", grid.best_params_)
print("Best Score :", grid.best_score_)

In [ ]:
# evaluating best model
best_model = grid.best_estimator_
y_pred_best = best_model.predict(x_test)
rmse_best = root_mean_squared_error(y_test,y_pred_best)
r2_best = r2_score(y_test,y_pred_best)

print("Model Evaluation(Best Model)")
print("Root Mean Squared Error :", round(rmse_best,4))
print("R2 Score :", round(r2_best,4))

In [ ]:
# plot epsilon values vs model prediction
epsilon = [0,0.1,0.5,0.7,1]
test_r2 = []
test_r2_best = []
for e in epsilon:
    base_model.set_params(svr__epsilon=e)
    best_model.set_params(svr__epsilon=e)
    base_model.fit(x_train,y_train)
    best_model.fit(x_train,y_train)
    base_pred = base_model.predict(x_test)
    best_pred = best_model.predict(x_test)
    test_r2.append(r2_score(y_test,base_pred))
    test_r2_best.append(r2_score(y_test,best_pred))

fig,axes = plt.subplots(1,2,figsize=(12,5))
axes[0].plot(epsilon,test_r2,marker="*",color="brown")
axes[0].set_ylabel("R2 Score")
axes[0].set_xlabel("Epsilon Values")
axes[0].set_title("Base Model")

axes[1].plot(epsilon,test_r2_best,marker="^",color="purple")
axes[1].set_ylabel("R2 Score")
axes[1].set_xlabel("Epsilon Values")
axes[1].set_title("Best Model")

plt.suptitle("Change in Epsilon Values vs Model Preidction")
plt.tight_layout()
plt.show()

In [ ]:
# plot c_values vs model prediction
c_values = [1,3,5,7,10,30,50,75]
test_r2 = []
test_r2_best = []
for c in c_values:
    base_model.set_params(svr__C=c)
    best_model.set_params(svr__C=c)
    base_model.fit(x_train,y_train)
    best_model.fit(x_train,y_train)
    base_pred = base_model.predict(x_test)
    best_pred = best_model.predict(x_test)
    test_r2.append(r2_score(y_test,base_pred))
    test_r2_best.append(r2_score(y_test,best_pred))

fig,axes = plt.subplots(1,2,figsize=(12,5))
axes[0].plot(c_values,test_r2,marker="*",color="skyblue")
axes[0].set_ylabel("R2 Score")
axes[0].set_xlabel("C Values")
axes[0].set_title("Base Model")

axes[1].plot(c_values,test_r2_best,marker="^",color="red")
axes[1].set_ylabel("R2 Score")
axes[1].set_xlabel("C Values")
axes[1].set_title("Best Model")

plt.suptitle("Change in C_values vs Model Preidction")
plt.tight_layout()
plt.show()